### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [21]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.metrics import Precision, Recall, AUC

In [7]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [8]:
data['Exited'].value_counts(normalize=True) * 100

Exited
0    79.63
1    20.37
Name: proportion, dtype: float64

In [9]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [10]:
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [11]:
X = data.drop('Exited', axis=1)
y = data['Exited']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [14]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [16]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",
                  metrics=['accuracy', Precision(name='precision'), 
                           Recall(name='recall'), AUC(name='roc-auc')])

    return model

In [17]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [18]:
# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64],
    'layers': [1, 2, 3],
    'epochs': [10, 20, 30]
}

In [19]:
# Perform grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='roc_auc',
    n_jobs=-1,
    cv=3,
    verbose=1
)

grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 27 candidates, totalling 81 fits


c:\Users\viren\anaconda3\envs\churn_modelling_venv\Lib\site-packages\sklearn\model_selection\_search.py:1234: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan]
  warnings.warn(
c:\Users\viren\anaconda3\envs\churn_modelling_venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\viren\anaconda3\envs\churn_modelling_venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 790us/step - accuracy: 0.7807 - loss: 0.4973 - precision: 0.3333 - recall: 0.0761 - roc-auc: 0.6486
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 751us/step - accuracy: 0.8112 - loss: 0.4342 - precision: 0.6807 - recall: 0.1387 - roc-auc: 0.7589
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 741us/step - accuracy: 0.8207 - loss: 0.4193 - precision: 0.7085 - recall: 0.2043 - roc-auc: 0.7795
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 753us/step - accuracy: 0.8291 - loss: 0.4073 - precision: 0.7152 - recall: 0.2681 - roc-auc: 0.7955
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 738us/step - accuracy: 0.8372 - loss: 0.3967 - precision: 0.7343 - recall: 0.3153 - roc-auc: 0.8082
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 748us/step - accuracy: 0.8415 - loss: 0.3875 - precision: 0.7401 - recall: 0.3423 - roc-auc: 0.8191
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 749us/step - accuracy: 0.8441 - loss: 0.3783 - precision: 0.7355 - recall: 0.3669 - roc-auc: 0.8282

In [22]:
# Model Performance Evaluation
keras_model = grid.best_estimator_.model_
y_pred = keras_model.predict(X_test)
y_pred = (y_pred > 0.4)

print("\nConfusion Matrix : \n\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy Score: \n\n", accuracy_score(y_test, y_pred))
print("\nF1-Score : \n\n", f1_score(y_test, y_pred))
print("\nClassification Report: \n\n", classification_report(y_test, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step

Confusion Matrix : 

 [[1496   97]
 [ 193  214]]

Accuracy Score: 

 0.855

F1-Score : 

 0.596100278551532

Classification Report: 

               precision    recall  f1-score   support

           0       0.89      0.94      0.91      1593
           1       0.69      0.53      0.60       407

    accuracy                           0.85      2000
   macro avg       0.79      0.73      0.75      2000
weighted avg       0.85      0.85      0.85      2000

